# Task 3

In [ ]:
import awswrangler as wr
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
GLUE_DATABASE = "classicmodels_star_g4"
sns.set_theme(style="whitegrid")

In [ ]:
sql_dim_products = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

df_products = wr.athena.read_sql_query(sql=sql_dim_products, database=GLUE_DATABASE)
df_products.head()

,product_id,product_name,product_line,product_vendor
0,S12_1666,1958 Setra Bus,Trucks and Buses,Welly Diecast Productions
1,S18_1589,1965 Aston Martin DB5,Classic Cars,Classic Metal Creations
2,S18_1749,1917 Grand Touring Sedan,Vintage Cars,Welly Diecast Productions
3,S18_1889,1948 Porsche 356-A Roadster,Classic Cars,Gearbox Collectibles
4,S18_2238,1998 Chrysler Plymouth Prowler,Classic Cars,Gearbox Collectibles


In [ ]:
# Venda total por país

sql_sales_by_country = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_sales_country = wr.athena.read_sql_query(sql=sql_sales_by_country, database=GLUE_DATABASE)
df_sales_country

/home/luusamp/gits/fgv-projetos-20261/assignment_1/task_3/grupo_4/aluno_luciano/task_3/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,country,total_sales
0,USA,3273280.05
1,Spain,1099389.09
2,France,1007374.02
3,Australia,562582.59
4,New Zealand,476847.01
5,UK,436947.44
6,Italy,360616.81
7,Finland,295149.35
8,Singapore,263997.78
9,Denmark,218994.92


In [ ]:
# Detalhamento por data, produto e país

sql_detail = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_detail = wr.athena.read_sql_query(sql=sql_detail, database=GLUE_DATABASE)
df_detail["full_date"] = pd.to_datetime(df_detail["full_date"])
df_detail.head()

/home/luusamp/gits/fgv-projetos-20261/assignment_1/task_3/grupo_4/aluno_luciano/task_3/.venv/lib/python3.14/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,full_date,product_line,product_name,country,total_sales
0,2005-03-02,Motorcycles,1997 BMW F650 ST,Singapore,3516.04
1,2003-10-22,Classic Cars,2001 Ferrari Enzo,Singapore,7406.08
2,2003-10-22,Classic Cars,1969 Ford Falcon,Singapore,4111.02
3,2003-10-22,Classic Cars,1998 Chrysler Plymouth Prowler,Singapore,3893.54
4,2003-10-22,Classic Cars,1992 Ferrari 360 Spider red,Singapore,7242.70


In [ ]:
# Dashboard 

date_min = df_detail["full_date"].min().date()
date_max = df_detail["full_date"].max().date()
countries = sorted(df_detail["country"].dropna().unique().tolist())
product_lines = sorted(df_detail["product_line"].dropna().unique().tolist())

date_start = widgets.DatePicker(description="Data início", value=date_min)
date_end = widgets.DatePicker(description="Data fim", value=date_max)
country_filter = widgets.Dropdown(options=["Todos"] + countries, value="Todos", description="País")
line_filter = widgets.Dropdown(options=["Todos"] + product_lines, value="Todos", description="Linha")
top_n = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    step=1,
    description="Top N",
)
out = widgets.Output()


def upt_dashboard(*_):
    with out:
        clear_output(wait=True)
        start = pd.Timestamp(date_start.value)
        end = pd.Timestamp(date_end.value)
        
        if start > end:
            start, end = end, start

        filtered = df_detail[
            (df_detail["full_date"] >= start)
            & (df_detail["full_date"] <= end)
        ]
        if country_filter.value != "Todos":
            filtered = filtered[filtered["country"] == country_filter.value]
        if line_filter.value != "Todos":
            filtered = filtered[filtered["product_line"] == line_filter.value]

        ranked = (
            filtered.groupby("product_name", as_index=False)["total_sales"]
            .sum()
            .sort_values("total_sales", ascending=False)
            .head(top_n.value)
        )

        if ranked.empty:
            print("Nenhum dado para os filtros selecionados.")
            return

        fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(ranked))))
        sns.barplot(
            data=ranked,
            y="product_name",
            x="total_sales",
            ax=ax,
            hue="product_name",
            legend=False,
            palette="viridis",
        )
        ax.set_xlabel("Vendas totais")
        ax.set_ylabel("Produto")
        ax.set_title(
            f"Top {top_n.value} produtos — {country_filter.value} / {line_filter.value}"
        )
        plt.tight_layout()
        plt.show()


controls = widgets.VBox(
    [
        widgets.HBox([date_start, date_end]),
        widgets.HBox([country_filter, line_filter, top_n]),
        out,
    ]
)

for w in (date_start, date_end, country_filter, line_filter, top_n):
    w.observe(upt_dashboard, names="value")
display(controls)
upt_dashboard()